# Curating a Consistent Dataset for Machine Learning

Welcome to the final dataset curation step! 

Right now, we have a bunch of waveform files that are roughly 40 to 50 seconds long. However, Machine Learning (ML) models are very strict: they generally prefer taking in inputs of the exact same size, over and over again.

Our goal is to take our longer waveforms and cleanly **slice** them into perfect 10-second bite-sized pieces. Even better, we want to anchor these pieces so that the earthquake wave (the P-wave) hits at the *exact same moment* (dead center, at 5.0 seconds) in every single file!

### Step 1: Loading Libraries and Setup
We import our trusty tools, including `json` which we will use to create "metadata" labels for our ML model.

In [1]:
import os
import glob
import csv
import json
from collections import defaultdict
from obspy import read, UTCDateTime

csv_file = 'filtered_picks_organized.csv'
input_dir = 'selected_quick_migrate_mseed'
output_dir = 'trimmed_and_consistent_mseed'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

### Step 2: Finding the "Anchor" point
If Station A recorded an earthquake, and Station B recorded the *same* earthquake, the wave probably hit them at slightly different times.

Therefore, we must slice Station A based on its *own* local arrival time, and Station B based on its *own* local arrival time. We group our CSV picks by a combination of `(event_id, station)`. If there are multiple wave phases recorded (like a P-wave and an S-wave), we find the absolute earliest one (the P-wave) to serve as the anchor for that trace!

In [2]:
station_event_picks = defaultdict(dict)

with open(csv_file, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        event_id = row['event_id']
        station = row['station']
        phase = row['phase']
        try:
            pick_time = UTCDateTime(row['pick_time'])
            station_event_picks[(event_id, station)][phase] = pick_time
        except Exception:
            continue
            
print(f"Identified {len(station_event_picks)} unique station-event pairs.")

Identified 370 unique station-event pairs.


### Step 3: Indexing the Input Files
Just like we did in previous scripts, we build a quick-lookup index of our available `.mseed` files so we don't have to scan the hard drive thousands of times.

In [3]:
input_files = glob.glob(os.path.join(input_dir, '*.mseed'))
mseed_index = defaultdict(list)

for filepath in input_files:
    basename = os.path.basename(filepath)
    parts = basename.split('.')
    if len(parts) >= 6:
        station = parts[1]
        try:
            mseed_index[station].append({
                'filepath': filepath,
                'network': parts[0],
                'station': station,
                'channel': parts[2],
                'start_time': UTCDateTime(parts[3]),
                'end_time': UTCDateTime(parts[4])
            })
        except Exception:
            pass

### Step 4: Slicing the Traces & Writing Metadata (The Magic!)
Here is how we create the ML dataset:
1. We take our "anchor" time (`ref_time`).
2. We define a 10-second window starting 5 seconds before the anchor, and ending 5 seconds after.
3. We use `trace.slice()` to chop the waveform file precisely on those timestamps.
4. We write out a JSON file. This JSON file acts like an answer key for the ML model, telling it exactly what channels to expect, and exactly what sample the P-wave and S-wave arrive at.

In [4]:
processed_windows = set() # To avoid doing the same work twice
success_count = 0

for (event_id, station), picks in station_event_picks.items():
    ref_time = min(picks.values())
    
    # 1. Define the 10-second slice window
    window_start = ref_time - 5.0
    window_end = ref_time + 5.0
    
    saved_channels = []
    saved_files = []
    network = "XX"
    sample_rate = 200.0
    
    if station in mseed_index:
        for file_info in mseed_index[station]:
            # Only slice if the file actually covers our 10s window
            if file_info['start_time'] <= window_start and file_info['end_time'] >= window_end:
                
                w_start_str = window_start.strftime('%Y%m%dT%H%M%S')
                w_end_str = window_end.strftime('%Y%m%dT%H%M%S')
                
                new_filename = f"{file_info['network']}.{file_info['station']}.{file_info['channel']}.{w_start_str}.{w_end_str}.mseed"
                new_filepath = os.path.join(output_dir, new_filename)
                
                if new_filename not in processed_windows:
                    try:
                        # 2. Slice the file!
                        st = read(file_info['filepath'])
                        tr_sliced = st[0].slice(starttime=window_start, endtime=window_end)
                        
                        # Save it if the slice was successful
                        if tr_sliced.stats.npts > 0:
                            tr_sliced.write(new_filepath, format="MSEED")
                            processed_windows.add(new_filename)
                            
                            saved_channels.append(file_info['channel'])
                            saved_files.append(new_filename)
                            network = file_info['network']
                            sample_rate = tr_sliced.stats.sampling_rate
                            success_count += 1
                    except Exception:
                        pass
                        
    # 3. Create the JSON Metadata Answer Key
    if saved_files:
        # We enforce a strict channel order (HH1, HH2, HHZ) so the ML model doesn't get confused
        channel_order = {"1": 0, "E": 0, "2": 1, "N": 1, "Z": 2}
        def sort_key(pair):
            return channel_order.get(pair[0][-1], 99)
            
        paired = list(zip(saved_channels, saved_files))
        paired.sort(key=sort_key)
        saved_channels = [p[0] for p in paired]
        saved_files = [p[1] for p in paired]
        
        # Compile the JSON dictionary
        metadata = {
            "event_id": event_id,
            "station": station,
            "network": network,
            "start_time": window_start.strftime('%Y-%m-%dT%H:%M:%S.%f') + 'Z',
            "sample_rate": sample_rate,
            "duration": 10.0,
        }
        
        # The math trick: Because window_start is exactly 5 seconds before the P-wave, 
        # the p_relative time will ALWAYS be exactly 5.0 seconds!
        if 'P' in picks:
            p_relative = picks['P'] - window_start
            metadata["p_arrival_sample"] = int(p_relative * sample_rate)
            metadata["p_arrival_time"] = p_relative
            
        if 'S' in picks:
            s_relative = picks['S'] - window_start
            metadata["s_arrival_sample"] = int(s_relative * sample_rate)
            metadata["s_arrival_time"] = s_relative
            
        metadata["channels"] = saved_channels
        metadata["files"] = {"mseed": saved_files}
        
        # Save the JSON file alongside the .mseed files
        json_filename = f"{network}.{station}.{w_start_str}.{w_end_str}.json"
        json_filepath = os.path.join(output_dir, json_filename)
        with open(json_filepath, 'w') as json_file:
            json.dump(metadata, json_file, indent=4)
            
print(f"Successfully processed {success_count} MiniSEED slices and generated JSON metadata!")

Successfully processed 1110 MiniSEED slices and generated JSON metadata!


---
**Congratulations!**
You have built a fully automated pipeline that can take raw seismology recordings, parse spreadsheets of earthquake arrivals, dynamically slice the data into 10-second windows anchored on the arrival times, and automatically generate JSON answer keys for a Neural Network.

Your dataset is perfectly clean, consistent, and ready for advanced Machine Learning research!